<a href="https://colab.research.google.com/github/parthsharma5575/Ai-Ml-GenAi/blob/main/Email_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Email Draft Agent — LangChain + Groq
### An AI Agent that researches a topic and writes a professional email

---

## How This Agent Works
```

##  Tools Used

| Tool | Purpose |
|---|---|
| `TavilySearch` | Searches the web for context on the topic |
| `format_email` | Custom tool that structures the email properly |
| `get_date` | Custom tool that fetches today's date |
| `ChatGroq` | LLaMA 3.3 70B — the reasoning brain |

---

In [ ]:
!pip install -q langchain langchain-groq langchain-community langchain-tavily langchain-core


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
import os
from getpass import getpass


os.environ["GROQ_API_KEY"] = getpass("Groq API Key:")
os.environ["TAVILY_API_KEY"] = getpass("Tavily API Key:")

Groq API Key:··········
Tavily API Key:··········


In [ ]:
from langchain_core.tools import tool

from langchain_tavily import TavilySearch
from datetime import datetime

In [ ]:
tavily_tool = TavilySearch(
    max_results=3,
    topic = 'general'

)

In [ ]:
@tool
def format_email(email_details:str) -> str:
  """
    Formats and assembles a professional email.
    Input should be a single string containing all email details in this exact format:

    SENDER: [sender name]
    RECIPIENT: [recipient name]
    SUBJECT: [email subject]
    BODY: [full email body text]

    Example:
    SENDER: Aryan Sharma
    RECIPIENT: Dr. Priya Mehta
    SUBJECT: Request for Meeting
    BODY: Dear Dr. Mehta, I hope this email finds you well...
    """

  from datetime import datetime

  today = datetime.now().strftime("%B %d, %Y")



  lines = email_details.strip().split("\n")
  sender    = ""
  recipient = ""
  subject   = ""
  body_lines = []
  in_body = False

  for line in lines:
        line = line.strip()
        if line.upper().startswith("SENDER:"):
            sender = line[7:].strip()
        elif line.upper().startswith("RECIPIENT:"):
            recipient = line[10:].strip()
        elif line.upper().startswith("SUBJECT:"):
            subject = line[8:].strip()
        elif line.upper().startswith("BODY:"):
            in_body = True
            first = line[5:].strip()
            if first:
                body_lines.append(first)
        elif in_body:
            body_lines.append(line)

  body = "\n".join(body_lines)

  email = f"""
    {'='*55}
    📧 DRAFTED EMAIL
    {'='*55}
    Date      : {today}
    From      : {sender}
    To        : {recipient}
    Subject   : {subject}
    {'─'*55}

    {body}

    {'='*55}
    """
  return email

In [ ]:
@tool
def get_today_date(dummy:str) -> str:

    """
    Returns today's date in a readable format.
    Use this when you need the current date for the email header.
    Input can be any string or left empty.
    """
    return datetime.now().strftime("%B %d, %Y")

In [ ]:
tools = [tavily_tool,format_email,get_today_date]

In [ ]:
from langchain_groq import ChatGroq
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_core.prompts import PromptTemplate


In [ ]:
llm = ChatGroq(


               model = "llama-3.3-70b-versatile",
               temperature = 0
)

In [ ]:
email_prompt = PromptTemplate.from_template("""
You are a professional email writing assistant.
Your job is to research a topic using web search and then write a polished,
professional email based on the user's request.

Always follow these steps:
1. Use get_today_date to get the current date
2. Use tavily_search to gather relevant and current information about the topic
3. Use format_email to assemble and return the final email

Make sure the email is:
- Professional and well-structured
- Relevant (based on real information from your search)
- Appropriately toned (formal, semi-formal, or friendly as requested)
- Concise — no unnecessary filler

You have access to the following tools:
{tools}

Use this EXACT format:

Question: the input question you must answer
Thought: think about what to do next
Action: the action to take, must be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (repeat Thought/Action/Action Input/Observation as needed)
Thought: I now have everything I need to write the email
Final Answer: [paste the complete formatted email here]

Begin!

Question: {input}
Thought: {agent_scratchpad}
""")

In [ ]:
agent = create_react_agent(

                           llm = llm,
                           tools = tools,
                           prompt = email_prompt
)

In [ ]:
agent_ececutor = AgentExecutor(

                               agent = agent,
                               tools = tools,
                               verbose=True,
                               max_iterations=8,
                               handle_parsing_errors=True
)

In [ ]:
llm.model_name

'llama-3.3-70b-versatile'

In [ ]:
responce = agent_ececutor.invoke({
    "input": """
    Write a formal email from Aryan Sharma (AI Engineer at TechCorp)
    to Dr. Priya Mehta (Head of Innovation at DataVentures) requesting
    a meeting to discuss partnering on an AI project related to
    large language models in healthcare.
    Tone: Formal and professional.
    """
})



> Entering new AgentExecutor chain...
To write a formal email to Dr. Priya Mehta, I first need to get the current date to include in the email header.

Action: get_today_date
Action Input: dummy string (can be any string or left empty)July 23, 2026Now that I have the current date, I need to gather relevant and current information about large language models in healthcare to make the email more specific and interesting.

Action: tavily_search
Action Input: large language models in healthcare{'query': 'large language models in healthcare', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC12189880', 'title': 'Large Language Models in Healthcare and Medical Applications: A Review', 'content': 'A **.gov** website belongs to an official government organization in the United States. # Large Language Models in Healthcare and Medical Applications: A Review. This paper provides a systematic and in-depth examination of larg

In [ ]:
print(responce["output"])

📧 DRAFTED EMAIL
Date      : July 23, 2026
From      : Aryan Sharma
To        : Dr. Priya Mehta
Subject   : Request for Meeting to Discuss AI Project in Healthcare
──────────────────────────────────────────────────────

Dear Dr. Mehta, I hope this email finds you well. As the Head of Innovation at DataVentures, I believe your expertise in healthcare technology would be invaluable in discussing a potential partnership between TechCorp and DataVentures on an AI project related to large language models in healthcare. Recent studies have shown the significant potential of large language models in transforming medical practice through advanced natural language processing capabilities, with applications in clinical decision support, medical education, diagnostics, and patient care. I would like to schedule a meeting with you to explore how our companies can collaborate on this project, addressing critical challenges in privacy, ethical deployment, and factual accuracy. Please let me know your

In [ ]:
# ============================================================
# STEP 6: Interactive Email Generator
# ============================================================
# Type your email request and the agent will research + draft it!
# Type 'quit' to exit | 'verbose on/off' to toggle reasoning steps

print("📧 Email Draft Agent — Interactive Mode")
print("=" * 55)
print("Describe the email you want written.")
print("Example: 'Write a formal email from John to his manager")
print("         requesting a 2-day work from home approval'")
print("Type 'quit' to exit | 'verbose on' or 'verbose off' to toggle steps")
print("=" * 55)

while True:
    print()
    user_input = input("📝 Your request: ").strip()

    # ── Exit ────────────────────────────────────────────────
    if not user_input:
        continue
    if user_input.lower() == "quit":
        print("\n👋 Goodbye! Happy emailing! 📬")
        break

    # ── Toggle verbose mode ──────────────────────────────────
    if user_input.lower() == "verbose on":
        agent_ececutor.verbose = True
        print("🔍 Verbose ON — you'll see the agent's thinking steps")
        continue
    if user_input.lower() == "verbose off":
        agent_ececutor.verbose = False
        print("🔇 Verbose OFF — showing final email only")
        continue

    # ── Generate Email ───────────────────────────────────────
    try:
        print("\n⏳ Agent is researching and drafting your email...\n")
        result = agent_ececutor.invoke({"input": user_input})
        print("\n" + "="*55)
        print("📬 YOUR DRAFTED EMAIL:")
        print("="*55)
        print(result["output"])

    except Exception as e:
        print(f"⚠️ Error: {e}")
        print("Please try rephrasing your request.")

📧 Email Draft Agent — Interactive Mode
Describe the email you want written.
Example: 'Write a formal email from John to his manager
         requesting a 2-day work from home approval'
Type 'quit' to exit | 'verbose on' or 'verbose off' to toggle steps

📝 Your request: 'Write a formal email from John to his manager          requesting a 2-day work from home approval'

⏳ Agent is researching and drafting your email...



> Entering new AgentExecutor chain...
To write a formal email from John to his manager requesting a 2-day work from home approval, I first need to get the current date to include in the email header.

Action: get_today_date
Action Input: dummy string (can be any string or left empty)July 23, 2026Next, I need to research the topic of work from home policies to ensure my email is well-informed and relevant. 

Action: tavily_search
Action Input: work from home policies and best practices{'query': 'work from home policies and best practices', 'follow_up_questions': None, 'a